In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

All libraries imported successfully


In [4]:
import os
os.chdir(os.path.expanduser("~/ransomware_xai_project"))
print(f"Working directory: {os.getcwd()}")
print(f"Files in data/: {os.listdir('data/')}")
df = pd.read_csv("data/MalMem2022.csv")
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Working directory: /Users/saneeshghimire/ransomware_xai_project
Files in data/: ['MalMem2022.csv', 'X_test_selected.csv', 'X_test_unscaled.csv', '.ipynb_checkpoints', 'X_train_unscaled.csv', 'y_train.csv', 'X_train_selected.csv', 'y_test.csv']
Dataset shape: (58596, 58)
Rows: 58596, Columns: 58


In [5]:
print("All columns:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col} ({df[col].dtype})")

All columns:
  0: pslist.nproc (int64)
  1: pslist.nppid (int64)
  2: pslist.avg_threads (float64)
  3: pslist.nprocs64bit (int64)
  4: pslist.avg_handlers (float64)
  5: dlllist.ndlls (int64)
  6: dlllist.avg_dlls_per_proc (float64)
  7: handles.nhandles (int64)
  8: handles.avg_handles_per_proc (float64)
  9: handles.nport (int64)
  10: handles.nfile (int64)
  11: handles.nevent (int64)
  12: handles.ndesktop (int64)
  13: handles.nkey (int64)
  14: handles.nthread (int64)
  15: handles.ndirectory (int64)
  16: handles.nsemaphore (int64)
  17: handles.ntimer (int64)
  18: handles.nsection (int64)
  19: handles.nmutant (int64)
  20: ldrmodules.not_in_load (int64)
  21: ldrmodules.not_in_init (int64)
  22: ldrmodules.not_in_mem (int64)
  23: ldrmodules.not_in_load_avg (float64)
  24: ldrmodules.not_in_init_avg (float64)
  25: ldrmodules.not_in_mem_avg (float64)
  26: malfind.ninjections (int64)
  27: malfind.commitCharge (int64)
  28: malfind.protection (int64)
  29: malfind.uniqueInje

In [6]:
print("Unique values in 'Class' column:")
print(df['Class'].value_counts())
print(f"\nTotal: {len(df)}")

Unique values in 'Class' column:
Class
Benign     29298
Malware    29298
Name: count, dtype: int64

Total: 58596


In [7]:
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"\nColumn: {col}")
        print(df[col].value_counts())

In [8]:
print("Category distribution:")
print(df['Category'].value_counts())

Category distribution:
Category
Benign                  29298
Spyware-Transponder      2410
Spyware-Gator            2200
Ransomware-Shade         2128
Ransomware-Ako           2000
Spyware-180solutions     2000
Spyware-CWS              2000
Trojan-Refroso           2000
Trojan-Scar              2000
Ransomware-Conti         1988
Trojan-Emotet            1967
Ransomware-Maze          1958
Trojan-Zeus              1950
Ransomware-Pysa          1717
Trojan-Reconyc           1570
Spyware-TIBS             1410
Name: count, dtype: int64


In [9]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)
print(f"Missing values:  {df.isnull().sum().sum()}")
print(f"Duplicate rows:  {df.duplicated().sum()}")

numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Infinite values: {np.isinf(df[numeric_cols]).sum().sum()}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Class balance:   {df['Class'].value_counts().to_dict()}")
print("=" * 50)
print("RESULT: Dataset is CLEAN" if df.isnull().sum().sum() == 0 else "WARNING: Issues found")

DATA QUALITY AUDIT
Missing values:  0
Duplicate rows:  534
Infinite values: 0
Numeric features: 55
Class balance:   {'Benign': 29298, 'Malware': 29298}
RESULT: Dataset is CLEAN


In [10]:
# Drop non-feature columns
drop_cols = ['Class', 'Category', 'Filename']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = (df['Class'] == 'Malware').astype(int)  # Benign=0, Malware=1

print(f"Features (X): {X.shape}")
print(f"Target (y):   {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\n0 = Benign, 1 = Malware")

Features (X): (58596, 55)
Target (y):   (58596,)
Target distribution:
Class
0    29298
1    29298
Name: count, dtype: int64

0 = Benign, 1 = Malware


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")
print(f"Test class dist:  {y_test.value_counts().to_dict()}")

Train set: (46876, 55)
Test set:  (11720, 55)
Train class dist: {0: 23438, 1: 23438}
Test class dist:  {1: 5860, 0: 5860}


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("StandardScaler applied")
print(f"Train mean (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Train std  (should be ~1): {X_train_scaled.std().mean():.6f}")
print("\nPreprocessing COMPLETE. Ready for feature selection.")

StandardScaler applied
Train mean (should be ~0): 0.000000
Train std  (should be ~1): 0.945465

Preprocessing COMPLETE. Ready for feature selection.
